In [ ]:
import pandas as pd
import plotly.express as px
from itertools import chain
import ast
import pandas as pd
import plotly.graph_objects as go

In [2]:
df = pd.read_csv("validation_summary_run2.csv")

In [3]:
df.head(5)

,iteration,model,total_matches,credible_matches,hallucination_perc_validation,hallucination_perc_generation,percent_credible,matched_genes,hallucinated_genes,hallucinated_genes_list,credible_pathways,credible_pathways_list,all_pathways_list
0,1,gpt-5-mini,2,0,100.000000,0.0,0.000000,62,0,[],0,[],"['Axon Guidance and Neurite Outgrowth', 'Myeli..."
1,1,gpt-5,7,0,100.000000,0.0,0.000000,62,0,[],0,[],"['Axon Guidance and Neurite Outgrowth', 'Myeli..."
2,1,gpt-4.1,55,20,63.636364,0.0,36.363636,62,0,[],6,"['Axon Guidance and Neurite Outgrowth', 'Myeli...","['Axon Guidance and Neurite Outgrowth', 'Myeli..."
3,1,gpt-4.1-mini,53,5,90.566038,0.0,9.433962,62,0,[],3,"['Axon Guidance and Neurite Outgrowth', 'Myeli...","['Axon Guidance and Neurite Outgrowth', 'Myeli..."
4,2,gpt-5-mini,0,0,0.000000,0.0,0.000000,71,0,[],0,[],['Axon Guidance and Semaphorin/Slit Signaling'...


## Data

In [4]:
it_col = ['model', 'iteration', 'credible_pathways', 'total_matches']
num_col = ['all_pathways_list']
lijst_model = ['gpt-5', 'gpt-5-mini', 'gpt-4.1', 'gpt-4.1-mini']

In [5]:
df_it = df[it_col]
for model in lijst_model:
    group_model = df.groupby("model")
    g_gpt = group_model.get_group(f"{model}")
    print(f"This is {model}----------------------------") 
    group = g_gpt[num_col]
    print(group)

This is gpt-5----------------------------
                                    all_pathways_list
1   ['Axon Guidance and Neurite Outgrowth', 'Myeli...
5   ['Axon Guidance and Semaphorin/Slit Signaling'...
9   ['Axon Guidance and Neurite Outgrowth', 'Synap...
13  ['Axon Guidance and Growth Cone Dynamics', 'Sy...
17  ['Axon Guidance and Synaptic Adhesion', 'Myeli...
21  ['Axon Guidance & Growth Cone Dynamics', 'Sema...
25  ['Axon Guidance and Semaphorin/Slit Signaling'...
29  ['Axon Guidance and Growth Cone Dynamics', 'Sy...
33  ['Axon Guidance and Neuronal Migration', 'Syna...
37  ['Semaphorin-Neuropilin/Plexin Signaling', 'Sl...
41  ['Axon Guidance and Neurite Outgrowth', 'Synap...
45  ['Synaptic Vesicle Trafficking and Plasticity'...
49  ['Axon Guidance', 'Semaphorin–Plexin Signaling...
This is gpt-5-mini----------------------------
                                    all_pathways_list
0   ['Axon Guidance and Neurite Outgrowth', 'Myeli...
4   ['Axon Guidance and Semaphorin/Slit Signali

In [6]:
print(df_it)

           model  iteration  credible_pathways  total_matches
0     gpt-5-mini          1                  0              2
1          gpt-5          1                  0              7
2        gpt-4.1          1                  6             55
3   gpt-4.1-mini          1                  3             53
4     gpt-5-mini          2                  0              0
5          gpt-5          2                  1              4
6        gpt-4.1          2                  8             66
7   gpt-4.1-mini          2                  4             58
8     gpt-5-mini          3                  0              0
9          gpt-5          3                  2              7
10       gpt-4.1          3                  5             71
11  gpt-4.1-mini          3                  3             65
12    gpt-5-mini          4                  0              0
13         gpt-5          4                  2              7
14       gpt-4.1          4                  5             64
15  gpt-

In [7]:
df_long = df_it.melt(
    id_vars=['model', 'iteration'],                    # Columns to keep
    value_vars=['total_matches', 'credible_pathways'], # Columns to 'melt'
    var_name='measurement_type',                       # Name of new category column
    value_name='amount'                                # Name of new values column
)

# Renaming for better legend
df_long['measurement_type'] = df_long['measurement_type'].replace({
    'total_matches': 'Total matches',
    'credible_pathways': 'Credible pathways'
})

print("--- 'Melted' DataFrame (Long Format) ---")
print(df_long.head())

--- 'Melted' DataFrame (Long Format) ---
          model  iteration measurement_type  amount
0    gpt-5-mini          1    Total matches       2
1         gpt-5          1    Total matches       7
2       gpt-4.1          1    Total matches      55
3  gpt-4.1-mini          1    Total matches      53
4    gpt-5-mini          2    Total matches       0


## Plots

Shows how many matched pathways the model has found (total_matches) and how many pathways are backed by real articles (credible pathways). 

In [8]:
fig_combined = px.box(
    df_long,
    x='model',                # Models on X-axis
    y='amount',               # Numbers on Y-axis
    color='measurement_type', # Makes the 'side-by-side' group
    points='all',
    height=600,
    title='Side by side: Total matches and credible pathways',
    labels={
        "model": "Model",
        "amount": "Pathways",
        "measurement_type": "Measurement type"
    }
)
fig_combined.show()

### Amount of credible citations per iteration

In [9]:
# make credible_count column
df['credible_count'] = df['percent_credible'] / 100 * df['total_matches']

# If credible_count is 0, set minimum bubble size to 0.1 to make it visible
df['credible_count_plot'] = df['credible_count'].apply(lambda x: max(x, 0.1))


fig = px.scatter(
    df,
    x='total_matches',
    y='percent_credible',
    size='credible_count_plot',
    color='model',
    hover_name='model',
    hover_data=['total_matches', 'credible_count'],
    title='Percentage credible citations vs total generated citations',
    labels={
        'total_matches': 'Total citations',
        'percent_credible': 'Percentage credible citations (%)',
        'credible_count': 'Number of credible citations',
        'model': 'Model'
    },
    size_max=40,
    category_orders={'model': ['gpt-5-mini', 'gpt-5', 'gpt-4.1', 'gpt-4.1-mini']}  # force order and legend
)

fig.show()


In [10]:
# fig = px.scatter(
#     df,
#     x='matched_genes',
#     y='percent_credible',
#     size='matched_genes',
#     color='model',
#     hover_name='model',
#     hover_data=['matched_genes', 'hallucinated_genes'],
#     title='Correct gene matches vs credible genes per model',
#     labels={
#         'matched_genes': 'Aantal correcte genen',
#         'percent_credible': 'Percentage credible genes (%)',
#         'model': 'Model'
#     },
#     size_max=40,
#     category_orders={'model': ['gpt-5-mini', 'gpt-5', 'gpt-4.1', 'gpt-4.1-mini']}  # force order and legend
# )

# fig.show()


In [11]:
# Calculate amount of credible and non-credible citations
df['credible_count'] = df['percent_credible'] / 100 * df['total_matches']
df['noncredible_count'] = df['total_matches'] - df['credible_count']


# Transform data to "long format" to make stacked bars 
df_long = df.melt(
    id_vars=['iteration', 'model'],
    value_vars=['credible_count', 'noncredible_count'],
    var_name='Type',
    value_name='Aantal'
)

# Labels for readability
df_long['Type'] = df_long['Type'].replace({
    'credible_count': 'Credible',
    'noncredible_count': 'Non-credible'
})

fig = px.bar(
    df_long,
    x='iteration',
    y='Aantal',
    color='Type',
    facet_col='model',  # one facet per model
    barmode='stack',
    title='Number of credible vs non-credible citations per iteration and model',
    labels={
        'iteration': 'Iteration',
        'Aantal': 'Number of citations',
        'Type': 'Match type'
    }
)

fig.show()

In [12]:
# df['credible_genes_count'] = df['matched_genes'] * df['percent_credible'] / 100
# df['noncredible_genes_count'] = df['matched_genes'] - df['credible_genes_count']

# df_long = df.melt(
#     id_vars=['iteration','model'],
#     value_vars=['credible_genes_count','noncredible_genes_count'],
#     var_name='Type',
#     value_name='Amount'
# )

# df_long['Type'] = df_long['Type'].replace({
#     'credible_genes_count':'Credible',
#     'noncredible_genes_count':'Non-credible'
# })

# fig = px.bar(
#     df_long,
#     x='iteration',
#     y='Amount',
#     color='Type',
#     facet_col='model',
#     barmode='stack',
#     title='Number of credible vs non-credible citations',
#     labels={
#         'iteration':'Iteration',
#         'Amount':'Number of citations',
#         'Type':'Type'
#     }
# )

# fig.show()


In [13]:
# fig_bar = px.bar(
#     df,
#     x='iteration',
#     y='credible_pathways',
#     color='model',
#     barmode='group', 
    
#     height=600,
#     title='Credible matches from model every iteration',
#     labels={
#         "iteration": "Iteration number",
#         "credible_pathways": "Credible Pathways found",
#         "model": "Model Type"
#     }
# )

# fig_bar.show()

## Halucination percentage boxplot

(non credible matches percentage)

In [14]:
fig_hal = px.box(
    df,
    x='model',
    y='hallucination_perc_validation',
    color='model',
    points='all',
    height = 800,
    title="Distribution hallucionation percentage of different models (amount of non credible articles)",
    labels={
        "model": "Model Type", 
        "hallucination_percentage": "Hallucinatiepercentage (%)" #
    }
)
fig_hal.show()

## Percentage credible matches

percent_credible = (credible_pathways / total_matches) * 100

In [15]:
fig_cred = px.box(
    df,
    x='model',
    y='percent_credible',
    color='model',
    points='all',
    height = 800,
    title="Percentage of credible articles",
    labels={
        "model": "Model Type", 
        "hallucination_percentage": "Percentage credible matches (%)" #
    }
)
fig_cred.show()

In [16]:
# Make python list of string column
df['credible_pathways_list'] = df['credible_pathways_list'].apply(
    lambda x: ast.literal_eval(x) if x and x != '[]' else []
)

print(df['credible_pathways_list'].head())

# Explode list zo that every pathway will be on a new row
df_exp = df.explode('credible_pathways_list')

# remove emtpy or NaN pathways
df_exp = df_exp[df_exp['credible_pathways_list'].notna() & (df_exp['credible_pathways_list'] != '')]

print(df_exp[['model','credible_pathways_list']].head())


0                                                   []
1                                                   []
2    [Axon Guidance and Neurite Outgrowth, Myelinat...
3    [Axon Guidance and Neurite Outgrowth, Myelinat...
4                                                   []
Name: credible_pathways_list, dtype: object
     model                             credible_pathways_list
2  gpt-4.1                Axon Guidance and Neurite Outgrowth
2  gpt-4.1       Myelination and Schwann Cell Differentiation
2  gpt-4.1  Synaptic Vesicle Cycling and Neurotransmitter ...
2  gpt-4.1  Extracellular Matrix Remodeling in Neural Deve...
2  gpt-4.1  Calcium-Dependent Mechanisms of Synaptic Plast...


In [17]:
# # Make sets of credible pathways per model
# models = ['gpt-5-mini', 'gpt-5', 'gpt-4.1', 'gpt-4.1-mini']
# model_sets = {m: set(df_exp[df_exp['model'] == m]['credible_pathways_list']) for m in models}

# # Make list of unique pathways. Chain adds all sets together. 
# all_pathways = list(set(chain.from_iterable(model_sets.values())))

# # Make dataframe with indicater per model if pathway was found
# df_upset = pd.DataFrame({'pathway': all_pathways})
# for m in models:
#     df_upset[m] = df_upset['pathway'].apply(lambda x: 1 if x in model_sets[m] else 0)

# # Plot heatmap
# fig = px.imshow(
#     df_upset.set_index('pathway').T,
#     labels=dict(x="Pathway", y="Model", color="Found (1 = yes)"),
#     title="Overview of credible pathways per model",
#     width=1500,  
#     height=700
# )
# fig.show()


In bovenstaande plot zijn alle credible matches van de 13 iteraties te zien. Geel betekent dat het model deze heeft gevonden en blauw dat het deze niet heeft gevonden. We zien weer dat gpt-5-mini dit het slechtst doet. Deze vindt geen een credible pathway. GPT-4.1 doet het weer het best, met maar 1 pathway die hij niet vindt. 

In [24]:
df = pd.read_csv("validation_summary_run2.csv")

# Aggregate per model
df_model = df.groupby("model").agg(
    avg_matched_genes=("matched_genes", "mean"),
    avg_total_citations=("total_matches", "mean")
).reset_index()

# Round 
df_model["avg_total_citations_text"] = df_model["avg_total_citations"].round(2)
df_model["avg_matched_genes_text"] = df_model["avg_matched_genes"].round(2)

# Make bar for generated citations
bar = go.Bar(
    x=df_model["model"],
    y=df_model["avg_total_citations"],
    text=df_model["avg_total_citations_text"],
    textposition="outside",
    name="Generated citations"
)

# Add line for matched genes
line = go.Scatter(
    x=df_model["model"],
    y=df_model["avg_matched_genes"],
    mode="lines+markers+text",
    name="Matched genes (ground truth)",
    line=dict(color="red", dash="dash"),
    marker=dict(size=8),
    text=df_model["avg_matched_genes_text"],
    textposition="top center"
)


fig = go.Figure(data=[bar, line])

fig.update_layout(
    title="Average number of matched genes and generated citations per model",
    xaxis_title="Model",
    yaxis_title="Average count",
    height=600
)

fig.show()
